# IMU vs Skeleton Joint Kinematics Mapping Validation

这个 notebook 用已经提取好的 IMU 关节运动学特征和 3D skeleton 上肢运动学特征做对比，用来验证之前推断的 Movella DOT IMU 身体段映射是否合理。

核心思路：

- 两套特征都按原始传感器 `time_s/frame_idx` 输出，采样间隔约为 `1/120s`。
- 对比同一录制、同一侧、同一关节的角度、角速度、0.35s range/std。
- 用 normalized time series、相关矩阵、best-lag correlation 检查 IMU 映射是否和 skeleton 空间运动一致。
- 重点看趋势，不强要求绝对角度完全一致，因为 IMU 四元数相对角和 skeleton 几何角的定义不完全相同。

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True

DATASET_ROOT = Path('/home/julius/datasets/seminar_object_detection')
IMU_ROOT = DATASET_ROOT / 'upper_feature_outcome' / 'imu_joint_kinematics_features'
SKEL_ROOT = DATASET_ROOT / 'upper_feature_outcome' / 'skeleton_upper_limb_kinematics_features'

IMU_INDEX = IMU_ROOT / 'imu_joint_kinematics_index.csv'
SKEL_INDEX = SKEL_ROOT / 'skeleton_upper_limb_kinematics_index.csv'

FULL_ID = 'IA_4_H_02_Brief_schreiben1'
START_S = None  # e.g. 10.0
END_S = None    # e.g. 40.0
SMOOTH_S = 0.25
LAG_LIMIT_S = 0.50

SENSOR_MAPPING = {
    'D422CD008603': 'left_forearm',
    'D422CD008CC7': 'right_hand',
    'D422CD009F54': 'left_hand',
    'D422CD009F5B': 'right_forearm',
    'D422CD009F60': 'right_upper_arm',
    'D422CD00A220': 'left_upper_arm',
    'D422CD00A795': 'trunk',
}
SENSOR_MAPPING

In [ ]:
imu_index = pd.read_csv(IMU_INDEX)
skel_index = pd.read_csv(SKEL_INDEX)

shared = sorted(set(imu_index['full_id']).intersection(skel_index['full_id']))
print('shared recordings:', len(shared))
print(shared[:10])

display(imu_index.head())
display(skel_index.head())

In [ ]:
def read_pair(full_id=FULL_ID, start_s=START_S, end_s=END_S):
    imu_row = imu_index.loc[imu_index['full_id'] == full_id].iloc[0]
    skel_row = skel_index.loc[skel_index['full_id'] == full_id].iloc[0]
    imu = pd.read_csv(imu_row['frame_feature_csv'])
    skel = pd.read_csv(skel_row['frame_features_csv'])

    if start_s is not None:
        imu = imu[imu['time_s'] >= start_s]
        skel = skel[skel['time_s'] >= start_s]
    if end_s is not None:
        imu = imu[imu['time_s'] <= end_s]
        skel = skel[skel['time_s'] <= end_s]

    imu = imu.sort_values('time_s').reset_index(drop=True)
    skel = skel.sort_values('time_s').reset_index(drop=True)
    merged = pd.merge_asof(
        imu,
        skel,
        on='time_s',
        direction='nearest',
        tolerance=1 / 240,
        suffixes=('_imu', '_skel'),
    )
    return imu, skel, merged, imu_row, skel_row

imu, skel, merged, imu_row, skel_row = read_pair()

print('full_id:', FULL_ID)
print('imu rows:', len(imu), 'skel rows:', len(skel), 'merged rows:', len(merged))
print('imu median dt:', imu['time_s'].diff().dropna().median())
print('skel median dt:', skel['time_s'].diff().dropna().median())
print('video columns in imu:', [c for c in imu.columns if 'video' in c][:5])
print('video columns in skel:', [c for c in skel.columns if 'video' in c][:5])
print('skeleton body id:', skel_row.get('body_id'))

## Column Pair Definition

IMU 的角度列来自相对四元数：`parent_segment^-1 * child_segment`。

Skeleton 的角度列来自 3D 关键点几何：shoulder/elbow/wrist 的向量夹角。

所以绝对值可能存在固定偏移或定义差异，更适合比较变化趋势、速度、窗口 range/std。

In [ ]:
JOINTS = ['shoulder', 'elbow', 'wrist']
SIDES = ['left', 'right']

ANGLE_PAIRS = {
    (side, joint): (f'{side}_{joint}_angle_deg', f'{side}_{joint}_angle')
    for side in SIDES
    for joint in JOINTS
}
VELOCITY_PAIRS = {
    (side, joint): (f'{side}_{joint}_angle_velocity_deg_s', f'{side}_{joint}_angle_velocity')
    for side in SIDES
    for joint in JOINTS
}
RANGE_PAIRS = {
    (side, joint): (f'{side}_{joint}_angle_range_0p35s_imu', f'{side}_{joint}_angle_range_0p35s_skel')
    for side in SIDES
    for joint in JOINTS
}
STD_PAIRS = {
    (side, joint): (f'{side}_{joint}_angle_std_0p35s_imu', f'{side}_{joint}_angle_std_0p35s_skel')
    for side in SIDES
    for joint in JOINTS
}

for label, pairs in [('angle', ANGLE_PAIRS), ('velocity', VELOCITY_PAIRS), ('range', RANGE_PAIRS), ('std', STD_PAIRS)]:
    missing = []
    for key, (imu_col, skel_col) in pairs.items():
        if imu_col not in merged.columns or skel_col not in merged.columns:
            missing.append((key, imu_col, skel_col))
    print(label, 'missing:', missing)

In [ ]:
def zscore(series):
    s = pd.to_numeric(series, errors='coerce')
    std = s.std(skipna=True)
    if not np.isfinite(std) or std == 0:
        return s * np.nan
    return (s - s.mean(skipna=True)) / std

def smooth_series(df, column, smooth_s=SMOOTH_S):
    s = pd.to_numeric(df[column], errors='coerce')
    if smooth_s is None or smooth_s <= 0:
        return s
    indexed = s.copy()
    indexed.index = pd.to_timedelta(df['time_s'], unit='s')
    return indexed.rolling(f'{smooth_s}s', min_periods=1, center=True).mean().to_numpy()

def pair_corr(df, imu_col, skel_col, smooth_s=SMOOTH_S):
    x = pd.Series(smooth_series(df, imu_col, smooth_s=smooth_s))
    y = pd.Series(smooth_series(df, skel_col, smooth_s=smooth_s))
    mask = x.notna() & y.notna()
    if mask.sum() < 20:
        return np.nan, int(mask.sum())
    return float(x[mask].corr(y[mask])), int(mask.sum())

def best_lag_corr(df, imu_col, skel_col, lag_limit_s=LAG_LIMIT_S, smooth_s=SMOOTH_S):
    dt = df['time_s'].diff().dropna().median()
    if not np.isfinite(dt) or dt <= 0:
        dt = 1 / 120
    max_lag = int(round(lag_limit_s / dt))
    x = pd.Series(smooth_series(df, imu_col, smooth_s=smooth_s))
    y = pd.Series(smooth_series(df, skel_col, smooth_s=smooth_s))
    rows = []
    for lag in range(-max_lag, max_lag + 1):
        yy = y.shift(lag)
        mask = x.notna() & yy.notna()
        corr = x[mask].corr(yy[mask]) if mask.sum() >= 20 else np.nan
        rows.append({'lag_samples': lag, 'lag_s': lag * dt, 'corr': corr, 'n': int(mask.sum())})
    out = pd.DataFrame(rows)
    valid = out.dropna(subset=['corr'])
    if valid.empty:
        return out, None
    best = valid.iloc[valid['corr'].abs().argmax()].to_dict()
    return out, best

def compare_pairs(df, pairs, metric_name):
    rows = []
    for (side, joint), (imu_col, skel_col) in pairs.items():
        corr, n = pair_corr(df, imu_col, skel_col)
        _, best = best_lag_corr(df, imu_col, skel_col)
        rows.append({
            'metric': metric_name,
            'side': side,
            'joint': joint,
            'imu_col': imu_col,
            'skel_col': skel_col,
            'same_time_corr': corr,
            'best_lag_s': best['lag_s'] if best else np.nan,
            'best_lag_corr': best['corr'] if best else np.nan,
            'n': n,
        })
    return pd.DataFrame(rows)

summary = pd.concat([
    compare_pairs(merged, ANGLE_PAIRS, 'angle'),
    compare_pairs(merged, VELOCITY_PAIRS, 'angle_velocity'),
    compare_pairs(merged, RANGE_PAIRS, 'range_0p35s'),
    compare_pairs(merged, STD_PAIRS, 'std_0p35s'),
], ignore_index=True)

display(summary.sort_values(['metric', 'side', 'joint']))

## Mapping Sanity Check Matrix

如果 IMU 映射正确，`left_elbow` IMU 通常应该更像 `left_elbow` skeleton，而不是右侧或其他关节。这里用角速度做矩阵，因为它比绝对角度更少受定义偏移影响。

In [ ]:
imu_velocity_cols = [f'{side}_{joint}_angle_velocity_deg_s' for side in SIDES for joint in JOINTS]
skel_velocity_cols = [f'{side}_{joint}_angle_velocity' for side in SIDES for joint in JOINTS]

corr_matrix = pd.DataFrame(index=imu_velocity_cols, columns=skel_velocity_cols, dtype=float)
for imu_col in imu_velocity_cols:
    for skel_col in skel_velocity_cols:
        corr, _ = pair_corr(merged, imu_col, skel_col)
        corr_matrix.loc[imu_col, skel_col] = corr

display(corr_matrix.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format('{:.2f}'))

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(corr_matrix.astype(float), cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr_matrix.index)))
ax.set_yticklabels(corr_matrix.index)
ax.set_title(f'IMU vs Skeleton angle velocity correlation: {FULL_ID}')
for i in range(corr_matrix.shape[0]):
    for j in range(corr_matrix.shape[1]):
        value = corr_matrix.iloc[i, j]
        text = '' if pd.isna(value) else f'{value:.2f}'
        ax.text(j, i, text, ha='center', va='center', fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()

In [ ]:
best_matches = []
for imu_col in corr_matrix.index:
    row = corr_matrix.loc[imu_col].dropna()
    if row.empty:
        continue
    best_col = row.abs().idxmax()
    expected = imu_col.replace('_deg_s', '')
    expected = expected.replace('_velocity', '_velocity')
    expected_skel = imu_col.replace('_angle_velocity_deg_s', '_angle_velocity')
    best_matches.append({
        'imu_col': imu_col,
        'expected_skeleton_col': expected_skel,
        'best_skeleton_col': best_col,
        'best_corr': row.loc[best_col],
        'expected_corr': row.get(expected_skel, np.nan),
        'best_is_expected': best_col == expected_skel,
    })

best_matches = pd.DataFrame(best_matches)
display(best_matches)
print('best-is-expected ratio:', best_matches['best_is_expected'].mean())

## Segment Motion Sanity Matrix

这个矩阵更贴近“传感器贴在哪个身体段”的验证：用 IMU 每个身体段的旋转强度 `abs_gyro_mag` 对比 skeleton 对应关键点/手部中心的运动强度。它不能替代关节角验证，但很适合发现左右手或上臂/前臂明显串位。

In [ ]:
IMU_SEGMENT_MOTION_COLS = [
    'trunk_abs_gyro_mag',
    'left_upper_arm_abs_gyro_mag',
    'left_forearm_abs_gyro_mag',
    'left_hand_abs_gyro_mag',
    'right_upper_arm_abs_gyro_mag',
    'right_forearm_abs_gyro_mag',
    'right_hand_abs_gyro_mag',
]
SKEL_MOTION_COLS = [
    'neck_speed',
    'left_shoulder_speed',
    'left_elbow_speed',
    'left_wrist_speed',
    'left_hand_center_speed',
    'right_shoulder_speed',
    'right_elbow_speed',
    'right_wrist_speed',
    'right_hand_center_speed',
]
EXPECTED_SEGMENT_TO_SKEL = {
    'trunk_abs_gyro_mag': 'neck_speed',
    'left_upper_arm_abs_gyro_mag': 'left_elbow_speed',
    'left_forearm_abs_gyro_mag': 'left_wrist_speed',
    'left_hand_abs_gyro_mag': 'left_hand_center_speed',
    'right_upper_arm_abs_gyro_mag': 'right_elbow_speed',
    'right_forearm_abs_gyro_mag': 'right_wrist_speed',
    'right_hand_abs_gyro_mag': 'right_hand_center_speed',
}

segment_corr = pd.DataFrame(index=IMU_SEGMENT_MOTION_COLS, columns=SKEL_MOTION_COLS, dtype=float)
for imu_col in IMU_SEGMENT_MOTION_COLS:
    for skel_col in SKEL_MOTION_COLS:
        corr, _ = pair_corr(merged, imu_col, skel_col)
        segment_corr.loc[imu_col, skel_col] = corr

display(segment_corr.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format('{:.2f}'))

segment_best = []
for imu_col in segment_corr.index:
    row = segment_corr.loc[imu_col].dropna()
    if row.empty:
        continue
    best_col = row.abs().idxmax()
    expected_col = EXPECTED_SEGMENT_TO_SKEL[imu_col]
    segment_best.append({
        'imu_segment_motion': imu_col,
        'expected_skeleton_motion': expected_col,
        'best_skeleton_motion': best_col,
        'expected_corr': row.get(expected_col, np.nan),
        'best_corr': row.loc[best_col],
        'best_is_expected': best_col == expected_col,
    })
segment_best = pd.DataFrame(segment_best)
display(segment_best)
print('segment best-is-expected ratio:', segment_best['best_is_expected'].mean())

fig, ax = plt.subplots(figsize=(11, 6))
im = ax.imshow(segment_corr.astype(float), cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(segment_corr.columns)))
ax.set_xticklabels(segment_corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(segment_corr.index)))
ax.set_yticklabels(segment_corr.index)
ax.set_title(f'IMU segment gyro vs Skeleton motion correlation: {FULL_ID}')
for i in range(segment_corr.shape[0]):
    for j in range(segment_corr.shape[1]):
        value = segment_corr.iloc[i, j]
        text = '' if pd.isna(value) else f'{value:.2f}'
        ax.text(j, i, text, ha='center', va='center', fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()

## Time Series Overlay

蓝色是 IMU，橙色是 skeleton。这里画 z-score 后的曲线，只看趋势是否同步。

In [ ]:
def plot_pair(df, side='left', joint='elbow', metric='angle_velocity'):
    if metric == 'angle':
        imu_col, skel_col = ANGLE_PAIRS[(side, joint)]
    elif metric == 'angle_velocity':
        imu_col, skel_col = VELOCITY_PAIRS[(side, joint)]
    elif metric == 'range_0p35s':
        imu_col, skel_col = RANGE_PAIRS[(side, joint)]
    elif metric == 'std_0p35s':
        imu_col, skel_col = STD_PAIRS[(side, joint)]
    else:
        raise ValueError(metric)

    x = zscore(pd.Series(smooth_series(df, imu_col)))
    y = zscore(pd.Series(smooth_series(df, skel_col)))
    corr, n = pair_corr(df, imu_col, skel_col)
    lag_table, best = best_lag_corr(df, imu_col, skel_col)

    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=False)
    axes[0].plot(df['time_s'], x, label=f'IMU {imu_col}', linewidth=1)
    axes[0].plot(df['time_s'], y, label=f'Skeleton {skel_col}', linewidth=1, alpha=0.85)
    title = f'{side} {joint} {metric}: same-time corr={corr:.3f}, n={n}' if np.isfinite(corr) else f'{side} {joint} {metric}'
    if best:
        title += f', best lag={best["lag_s"]:.3f}s corr={best["corr"]:.3f}'
    axes[0].set_title(title)
    axes[0].set_ylabel('z-score')
    axes[0].legend(loc='upper right')

    axes[1].plot(lag_table['lag_s'], lag_table['corr'], color='black')
    axes[1].axvline(0, color='gray', linestyle='--')
    if best:
        axes[1].axvline(best['lag_s'], color='red', linestyle='--')
    axes[1].set_xlabel('skeleton shift lag (s)')
    axes[1].set_ylabel('corr')
    axes[1].set_title('Lag correlation')
    plt.tight_layout()

plot_pair(merged, side='left', joint='elbow', metric='angle_velocity')
plot_pair(merged, side='right', joint='elbow', metric='angle_velocity')
plot_pair(merged, side='left', joint='wrist', metric='range_0p35s')
plot_pair(merged, side='right', joint='wrist', metric='range_0p35s')

## Batch Scan Across Recordings

单条录制可能受遮挡、skeleton 缺点、动作类型影响。这个 cell 可以扫多条 shared recordings，统计 expected mapping 的平均相关性。

In [ ]:
def score_recording(full_id):
    _, _, df, _, _ = read_pair(full_id=full_id, start_s=None, end_s=None)
    rows = []
    for metric_name, pairs in [('angle_velocity', VELOCITY_PAIRS), ('range_0p35s', RANGE_PAIRS), ('std_0p35s', STD_PAIRS)]:
        for (side, joint), (imu_col, skel_col) in pairs.items():
            corr, n = pair_corr(df, imu_col, skel_col)
            rows.append({
                'full_id': full_id,
                'metric': metric_name,
                'side': side,
                'joint': joint,
                'corr': corr,
                'abs_corr': abs(corr) if np.isfinite(corr) else np.nan,
                'n': n,
            })
    return rows

BATCH_LIMIT = 20
batch_rows = []
for full_id in shared[:BATCH_LIMIT]:
    try:
        batch_rows.extend(score_recording(full_id))
    except Exception as exc:
        print('failed:', full_id, exc)

batch = pd.DataFrame(batch_rows)
display(batch.head())

pivot = batch.pivot_table(
    index=['metric', 'side', 'joint'],
    values=['corr', 'abs_corr', 'n'],
    aggfunc={'corr': 'mean', 'abs_corr': 'mean', 'n': 'sum'},
).reset_index().sort_values(['metric', 'side', 'joint'])
display(pivot)

## Interpretation Notes

- 如果 expected same-side/same-joint correlation 在多数录制里比 cross-side/cross-joint 更强，说明 IMU 身体段映射大方向是对的。
- `angle_velocity` 通常比 absolute `angle` 更适合验证映射，因为两套角度定义不同。
- wrist 的一致性可能弱于 elbow/shoulder：skeleton 手指点容易缺失，IMU 手部传感器测的是手段姿态，skeleton wrist angle 用的是前臂到手指中心的几何夹角。
- 如果某个 IMU 列总是和另一侧 skeleton 更相关，需要回头检查左右传感器映射。
- 如果某个 IMU joint 总是和相邻 skeleton joint 更相关，需要检查 parent/child segment 是否接反或身体段名称是否错位。